# CV and NLP Model Evaluation

Evaluate the local ConvNeXt damage checkpoint metadata and run repeatable baseline metrics for image damage classes and social sentiment.

In [ ]:
from pathlib import Path
import zipfile

CHECKPOINT = Path("../models_CV/week12_convnext_tiny_gated_ce_best.pt.zip").resolve()
print(CHECKPOINT)
print("exists:", CHECKPOINT.exists())

if CHECKPOINT.exists():
    with zipfile.ZipFile(CHECKPOINT) as zf:
        names = zf.namelist()
    print("archive files:", len(names))
    print("first files:", names[:8])

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

classes = ["no_damage", "minor", "major", "destroyed"]
rng = np.random.default_rng(42)
y_true = rng.choice(classes, size=500, p=[0.45, 0.25, 0.2, 0.1])
y_pred = rng.choice(classes, size=500, p=[0.4, 0.28, 0.22, 0.1])

print(classification_report(y_true, y_pred, labels=classes, zero_division=0))
print(confusion_matrix(y_true, y_pred, labels=classes))

In [ ]:
import sys
sys.path.append(str(Path("../backend").resolve()))

try:
    from app.ml.segmentation import get_segmentation_model
    model = get_segmentation_model()
    sample = rng.integers(0, 255, size=(512, 512, 3), dtype=np.uint8)
    features = model.predict(sample)
    print("mode:", getattr(model, "inference_mode", "unknown"))
    print("checkpoint:", getattr(model, "checkpoint_path", None))
    print("features:", len(features))
    print(features[0]["properties"] if features else {})
except Exception as exc:
    print("Backend CV model import skipped:", exc)

## Sentiment Baseline Evaluation

In [ ]:
examples = [
    ("Families are safe at the shelter and volunteers are helping.", "positive"),
    ("The road collapsed and people are trapped without water.", "negative"),
    ("Officials will provide another update at noon.", "neutral"),
    ("We are terrified by the flooding near the hospital.", "negative"),
    ("Relief teams rescued residents from the damaged area.", "positive"),
]

try:
    from app.ml.sentiment import get_sentiment_analyzer
    analyzer = get_sentiment_analyzer()
    y_true = [label for _, label in examples]
    y_pred = [analyzer.analyze(text)["sentiment"] for text, _ in examples]
    print("mode:", analyzer.mode)
    print(classification_report(y_true, y_pred, labels=["positive", "negative", "neutral"], zero_division=0))
except Exception as exc:
    print("Backend sentiment import skipped:", exc)